In [1]:
import spikeinterface.full as si
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

  # ── SET YOUR FORMAT HERE ───────────────────────────────────────
  FORMAT = 'spikeglx'   # 'spikeglx' | 'openephys' | 'mcs'

  # ── PATHS — fill in the one matching your format ───────────────
  spikeglx_folder  = Path(r'C:\Users\labuser\Ilaria\Project\small_rec\2142_g0\2142_g0_imec0')
  openephys_folder = Path(r'C:\path\to\openephys\session_folder')
  mcs_file         = Path(r'C:\path\to\recording.h5')   # or .raw

  base_folder = Path(r'C:\Users\labuser\Ilaria\Project\processed\2142_output')


C:\Users\labuser\miniforge3\envs\si_pipeline\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
 # ── LOAD ──────────────────────────────────────────────────────
  if FORMAT == 'spikeglx':
      raw_rec = si.read_spikeglx(spikeglx_folder,stream_name='imec0.ap', load_sync_channel=False)

  elif FORMAT == 'openephys':
      # run this first to find the correct stream_name for your session:
      # print(si.get_neo_streams('openephys', openephys_folder))
      raw_rec = si.read_openephys(openephys_folder, stream_name='Signals CH', load_sync_channel=False)

  elif FORMAT == 'mcs':
      # .h5 exported from Multi Channel DataManager:
      raw_rec = si.read_mcsh5(mcs_file)
      # for .raw binary files use instead:
      # raw_rec = si.read_mcsraw(mcs_file)

  print(raw_rec)

SpikeGLXRecordingExtractor: 384 channels - 30000.386555 Hz - 1 segments - 120,001 samples - 4.00s 
                            int16 dtype - 87.89 MiB


C:\Users\labuser\miniforge3\envs\si_pipeline\lib\site-packages\spikeinterface\extractors\neoextractors\spikeglx.py:99: UserWarning: Unable to find inter-sample shifts in the Neuropixels probe metadata. The sample shifts will not be loaded. 
  sample_shifts = get_neuropixels_sample_shifts_from_probe(probe, stream_name=self.stream_name)


In [5]:
  # ── PREPROCESSING (unchanged for all formats) ──────────────────
  rec1 = si.highpass_filter(raw_rec, freq_min=300.)
  bad_channel_ids, _ = si.detect_bad_channels(rec1)
  rec2 = rec1.remove_channels(bad_channel_ids)
  # phase_shift skipped — only relevant for Neuropixels ADC multiplexing
  rec = si.common_reference(rec2, operator="median", reference="global")
  print(rec)
# Should still show 384 ch (minus any bad channels)

CommonReferenceRecording: 383 channels - 30000.386555 Hz - 1 segments - 120,001 samples - 4.00s 
                          int16 dtype - 87.66 MiB


In [ ]:
#SPIKE SORTING

# save binary — keep, but change n_jobs and folder
job_kwargs = dict(
    n_jobs=4,                         
    chunk_duration='1s',
    progress_bar=True)
rec = rec.save(
    folder=base_folder / 'preprocess',
    format='binary',
    **job_kwargs)

# run sorter 
sorting = si.run_sorter(
    'kilosort4', rec,
    folder=base_folder / 'kilosort_output',
    remove_existing_folder=True, verbose=True)
print(sorting)